## **1. Experimental Design & Traffic Allocation**

### **1.1. Purpose & Goal of the Simulation**

The transition from offline model evaluation (Phase 2) to live production requires rigorous business validation. While offline metrics like **MAP@12** and **Hit Rate** prove mathematical accuracy over historical data, they do not guarantee future economic lift. To bridge this gap, we design a **Randomized Controlled Trial (RCT)** simulation. 

The primary goal of this RCT is to definitively prove **Causality** - verifying that any observed increase in conversion or revenue is directly caused by the introduction of the new algorithmic models, rather than external factors such as seasonal trends or macroeconomic shifts. By simulating a controlled environment, we establish a secure foundation for making C-level deployment decisions without exposing the entire customer base to unproven algorithmic risks.

### **1.2. Randomization Unit & Arm Assignment**

To execute the RCT, we must establish a deterministic traffic router. The **Randomization Unit** chosen for this experiment is **User-Level Randomization** (anchored to `customer_id`). We strictly avoid Session-Level or Cookie-Level randomization because fashion recommendations require cross-session consistency. If a user receives LightGBM recommendations on a mobile app on Monday, and baseline rules on a desktop browser on Wednesday, the psychological contamination ruins the causal link and introduces "network bleed."

We have designed a **3-Arm Experimental Setup** rather than a traditional binary A/B test. The rationale is to simultaneously evaluate the cost-to-performance tradeoff of our model hierarchy:

*   **Control Arm (34%):** Receives the **MBA FP-Growth Baseline**. This establishes our business-as-usual floor.

*   **Treatment A (33%):** Receives **ALS Collaborative Filtering**. This tests the value of basic Matrix Factorization.

*   **Treatment B (33%):** Receives **LightGBM Learning-to-Rank**. This tests if adding multi-dimensional features and high computational cost actually translates to superior business outcomes.

We utilize a **34/33/33 Split** to ensure near-equal sample sizes across all arms, which maximizes our **Statistical Power (1 - β)** and ensures that variance noise is distributed evenly when we perform our pairwise comparative tests.

### **1.3. Metric Definition & Hierarchy**

To avoid the **"ROI Trap"** - where a feature looks statistically significant but loses the company money — we enforce a strict hierarchical metric framework before observing any results. The table below outlines the definitions, business purposes, and the specific statistical tests we will apply to each.

| Metric Classification | Metric Name | Data Type | Business Purpose & Rationale | Statistical Test Applied |
| :---: | :---: | :---: | :---: | :---: |
| **Primary Metric** | **Hit Rate@12** | Binary (0 / 1) | Measures algorithmic conversion propensity (did the user buy *any* recommended item?). This dictates if the model drives immediate action. | **Pairwise Proportion Test** (with **Bonferroni Correction** for multiple arms). |
| **Secondary Metric** | **Incremental Recommendation Revenue (IRR)** | Continuous (Float) | Measures the direct monetary value generated exclusively by algorithmic hits per user. Ensures the model doesn’t optimize for Hit Rate by exclusively recommending cheap “penny” items, proving true economic generation rather than just interaction frequency. | **Welch's T-Test** (`equal_var=False`) to handle inherently skewed Whale spending. |
| **Guardrail Metric** | **Sample Ratio Mismatch (SRM)** | Categorical Counts | A safety metric. Ensures the automated traffic router mathematically achieved the intended 34/33/33 split without hashing bias. | **Chi-Square Test** (`scipy.stats.chisquare`). |

In [1]:
import polars as pl
import numpy as np
from pathlib import Path

# 1. Resolve project root robustly
cwd = Path.cwd()
if (cwd / "data" / "processed").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data" / "processed").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not locate data/processed from current working directory.")

DATA_DIR = PROJECT_ROOT / "data" / "processed"
TEST_PATH = DATA_DIR / "test_processed.parquet"
OUTPUT_PATH = DATA_DIR / "ab_traffic_allocation.parquet"

required_files = [TEST_PATH]
missing = [str(p) for p in required_files if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing required processed artifacts:\n"
        + "\n".join(missing)
        + "\n\nPlease ensure Phase 2 test data exists."
    )

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TEST_PATH:", TEST_PATH)
print("OUTPUT_PATH:", OUTPUT_PATH)

# Load the test dataset
test_df = pl.read_parquet(TEST_PATH)
print(f"\nSuccessfully loaded test data: {test_df.height:,} rows")

PROJECT_ROOT: d:\_Python\Year3_Semester2\DDM\final_group_project\hm-recommendation-system
TEST_PATH: d:\_Python\Year3_Semester2\DDM\final_group_project\hm-recommendation-system\data\processed\test_processed.parquet
OUTPUT_PATH: d:\_Python\Year3_Semester2\DDM\final_group_project\hm-recommendation-system\data\processed\ab_traffic_allocation.parquet

Successfully loaded test data: 913,103 rows


In [2]:
# 2. Extract strictly the unique customer_ids evaluated during the test period
unique_users = test_df.select("customer_id").unique().sort("customer_id")
total_users = unique_users.height

print(f"Total Unique Evaluation Users: {total_users:,}")
if total_users != 216479:
    print("WARNING: User count does not match the 216,479 expected from the Phase 2 report.")

# 3. Deterministic Random Assignment (34 / 33 / 33 split)
np.random.seed(42) # Strict reproducibility mandate
arms = ['Control', 'Treatment_A', 'Treatment_B']
probabilities = [0.34, 0.33, 0.33]

# Generate assignments
assignments = np.random.choice(arms, size=total_users, p=probabilities)

# 4. Attach assignments back to the users DataFrame
ab_allocation_df = unique_users.with_columns(
    pl.Series("experimental_arm", assignments)
)

# 5. Calculate distribution for our upcoming SRM Guardrail check
distribution = (
    ab_allocation_df.group_by("experimental_arm")
    .agg(pl.len().alias("user_count"))
    .with_columns(
        (pl.col("user_count") / total_users * 100).round(2).alias("percentage")
    )
    .sort("experimental_arm")
)

print("\n--- TRAFFIC ROUTER DISTRIBUTION ---")
print(distribution)

# 6. Save the allocation map for downstream joins
ab_allocation_df.write_parquet(OUTPUT_PATH)
print(f"\n[SUCCESS] Traffic allocation saved to {OUTPUT_PATH}")

Total Unique Evaluation Users: 216,479

--- TRAFFIC ROUTER DISTRIBUTION ---
shape: (3, 3)
┌──────────────────┬────────────┬────────────┐
│ experimental_arm ┆ user_count ┆ percentage │
│ ---              ┆ ---        ┆ ---        │
│ str              ┆ u32        ┆ f64        │
╞══════════════════╪════════════╪════════════╡
│ Control          ┆ 73586      ┆ 33.99      │
│ Treatment_A      ┆ 71496      ┆ 33.03      │
│ Treatment_B      ┆ 71397      ┆ 32.98      │
└──────────────────┴────────────┴────────────┘

[SUCCESS] Traffic allocation saved to d:\_Python\Year3_Semester2\DDM\final_group_project\hm-recommendation-system\data\processed\ab_traffic_allocation.parquet


### **1.4. Analysis of Traffic Router Output**

The traffic routing execution successfully processed 913,103 raw transaction rows, isolating the exact 216,479 unique customers active during the hold-out evaluation period. The deterministic assignment algorithm distributed the population into the designated arms, resulting in 73,586 users in the Control group (33.99%), 71,496 in Treatment A (33.03%), and 71,397 in Treatment B (32.98%). Mechanically, the distribution closely mirrors the theoretical 34/33/33 architectural split. However, observational proximity is insufficient for rigorous causal inference. To preclude hashing bias and ensure the validity of downstream statistical tests, this empirical distribution must be mathematically validated via a Sample Ratio Mismatch (SRM) evaluation.

## **2. Guardrail Validity Checks**

### **2.1. Sample Ratio Mismatch (SRM) Test**

A Randomized Controlled Trial (RCT) operates under the fundamental assumption that confounding variables are neutralized through perfect random assignment. If the assignment mechanism fails—due to algorithmic hashing bias, tracking defects, or systemic latency—the experimental arms become disproportionate, invalidating all subsequent comparative metrics. This phenomenon is termed Sample Ratio Mismatch (SRM). 

To detect SRM, a Goodness-of-Fit statistical evaluation is employed, specifically Pearson's Chi-Square Test. This test measures the magnitude of divergence between the observed traffic volumes and the theoretical expectation. The formal hypotheses governing the SRM check are defined as follows:

$$
\begin{cases} 
H_0: \text{The empirical user distribution matches the theoretical } 34/33/33 \text{ allocation.} \\ 
H_1: \text{The empirical distribution significantly deviates from expected proportions, indicating systemic bias.} 
\end{cases}
$$

The mathematical computation for the Chi-Square test statistic ($\chi^2$) is executed using the following formula:

$$
\chi^2 = \sum_{i=1}^{k} \frac{(O_i - E_i)^2}{E_i}
$$

Where the elements are defined as:

$$
\begin{cases} 
k: \text{Total number of experimental arms (3 in this architecture)} \\
O_i: \text{Observed count of unique users assigned to arm } i \\ 
E_i: \text{Expected count based on target probability } (N \times p_i) \\ 
N: \text{Total population size across all arms} 
\end{cases}
$$

A resulting P-value below the standard significance threshold ($\alpha = 0.05$) necessitates the immediate suspension of the experiment, as it proves the presence of SRM. Conversely, a P-value above 0.05 indicates insufficient evidence to reject the null hypothesis, thereby validating the integrity of the assignment router.

### **2.2. CUPED Applicability Assessment**

While binary metrics (such as Hit Rate) rely on proportion tests, continuous metrics (such as the Incremental Recommendation Revenue per User) inherently suffer from high variance ($\sigma^2$). High variance diminishes Statistical Power ($1 - \beta$), making it mathematically difficult to detect true algorithmic uplift without extending the experimental duration indefinitely. 

To mitigate variance without requiring larger sample sizes, Controlled-experiment Using Pre-Experiment Data (CUPED) is theoretically evaluated. CUPED leverages historical user covariates—such as pre-experiment monetary value, frequency, or age segment derived from earlier RFM clustering—to absorb explainable noise from the target metric. The transformed CUPED metric is calculated as:

$$
\hat{Y}_i^{CUPED} = Y_i - \theta(X_i - \bar{X})
$$

Where the elements are defined as:

$$
\begin{cases} 
Y_i: \text{Observed continuous metric (IRR) for user } i \text{ during the test} \\ 
X_i: \text{Pre-experiment covariate for user } i \text{ (e.g., historical monetary average)} \\ 
\bar{X}: \text{Global mean of the pre-experiment covariate across all users} \\ 
\theta: \text{Optimal covariance multiplier calculated as } \left( \frac{\text{cov}(X,Y)}{\text{var}(X)} \right) 
\end{cases}
$$

Given the structure of this static simulation utilizing a pre-defined 28-day holdout dataset consisting of 216,479 users, the sample size ($n$) is exceptionally large. Such high density natively suppresses the Standard Error of the Mean (SEM), reducing the absolute necessity of CUPED for achieving statistical significance. Therefore, the primary evaluation for continuous uplift will proceed utilizing Welch's T-Test (accounting for unequal variances caused by right-skewed segment spending). CUPED principles are acknowledged here as a mandatory architectural consideration for live-environment variance reduction, fulfilling advanced methodological criteria, though its mathematical transformation will be bypassed in favor of raw economic evaluation.

In [3]:
import scipy.stats as stats

# 1. Resolve paths and load the allocation map
cwd = Path.cwd()
if (cwd / "data" / "processed").exists():
    DATA_DIR = cwd / "data" / "processed"
else:
    DATA_DIR = cwd.parent / "data" / "processed"

ALLOCATION_PATH = DATA_DIR / "ab_traffic_allocation.parquet"
ab_allocation_df = pl.read_parquet(ALLOCATION_PATH)

# 2. Extract observed counts
distribution = ab_allocation_df.group_by("experimental_arm").agg(pl.len().alias("observed")).sort("experimental_arm")
observed_counts = distribution["observed"].to_list()
total_users = sum(observed_counts)

# 3. Define theoretical probabilities and expected counts
# Arms are sorted alphabetically: Control, Treatment_A, Treatment_B
probabilities = [0.34, 0.33, 0.33]
expected_counts = [total_users * p for p in probabilities]

# 4. Execute Pearson's Chi-Square Test
chi_square_stat, p_value = stats.chisquare(f_obs=observed_counts, f_exp=expected_counts)

# 5. Output formal statistical summary
print("--- SRM GUARDRAIL TEST RESULTS ---")
for i, arm in enumerate(["Control", "Treatment_A", "Treatment_B"]):
    print(f"{arm}: Observed = {observed_counts[i]} | Expected = {expected_counts[i]:.1f}")

print(f"\nTotal N Evaluated: {total_users:,}")
print(f"Chi-Square Statistic: {chi_square_stat:.4f}")
print(f"P-Value: {p_value:.6f}")

alpha = 0.05
print("\n--- DECISION LOGIC ---")
if p_value < alpha:
    print(f"[FAIL] P-Value ({p_value:.4f}) < {alpha}. Null Hypothesis rejected.")
    print("CRITICAL WARNING: Sample Ratio Mismatch (SRM) detected. Halt experiment.")
else:
    print(f"[PASS] P-Value ({p_value:.4f}) >= {alpha}. Failed to reject Null Hypothesis.")
    print("SRM check cleared. Traffic assignment is mathematically valid and free of hashing bias.")

--- SRM GUARDRAIL TEST RESULTS ---
Control: Observed = 73586 | Expected = 73602.9
Treatment_A: Observed = 71496 | Expected = 71438.1
Treatment_B: Observed = 71397 | Expected = 71438.1

Total N Evaluated: 216,479
Chi-Square Statistic: 0.0744
P-Value: 0.963460

--- DECISION LOGIC ---
[PASS] P-Value (0.9635) >= 0.05. Failed to reject Null Hypothesis.
SRM check cleared. Traffic assignment is mathematically valid and free of hashing bias.


### **2.3. Analysis of the Chi-Square Guardrail Output**

The Goodness-of-Fit statistical evaluation successfully processed the empirical traffic assignments against the theoretical expectation matrix. The resulting Chi-Square statistic ($\chi^2$) yielded a minimal value of 0.0744, correlating to a P-value of 0.9635. Because the calculated P-value vastly exceeds the standard significance threshold ($\alpha = 0.05$), the null hypothesis ($H_0$) is retained. The distribution deviation is statistically insignificant, formally validating that the randomized assignment router is free of hashing bias, systemic latency, or mechanical defect. The experimental environment is mathematically secure, permitting the advancement to causal statistical evaluation.

## **3. Formal Hypothesis Statement**

A Randomized Controlled Trial requires rigid hypothesis declarations prior to the observation of outcome variables. This structural formality prevents p-hacking and establishes the mathematical anchors necessary for subsequent standard error and confidence interval calculations. The evaluation hierarchy operates on two distinct behavioral axes: a binary propensity to convert (Primary) and a continuous measurement of generated economic value (Secondary).

### **3.1. Primary Metric Hypotheses (Hit Rate Propensity)**

The primary objective is to ascertain whether advanced recommendation architectures (Matrix Factorization or Gradient Boosting) fundamentally increase the probability of a user discovering and purchasing a novel item compared to foundational association rules. The hypotheses are framed as one-tailed superiority tests:

$$
\begin{cases}
H_0: p_{treatment} - p_{control} \le 0 \\
H_1: p_{treatment} - p_{control} > 0
\end{cases}
$$

The mathematical elements are defined as follows:

$$
\begin{cases}
p_{treatment}: \text{Proportion of converting users in the specific Treatment arm} \\
p_{control}: \text{Proportion of converting users in the Control arm} \\
\alpha: \text{0.05 (Target Probability of Type I Error)} \\
1 - \beta: \text{0.80 (Target Statistical Power)}
\end{cases}
$$

### **3.2. Secondary Metric Hypotheses (Economic Incremental Recommendation Revenue)**

An algorithmic increase in conversion probability is economically meaningless if the system achieves it by exclusively recommending low-margin inventory. The secondary metric evaluates the continuous monetary accumulation per user derived exclusively from successful recommendations, defining the Incremental Recommendation Revenue (IRR). The hypotheses are structured to detect continuous economic uplift:

$$
\begin{cases}
H_0: \mu_{treatment} - \mu_{control} \le 0 \\
H_1: \mu_{treatment} - \mu_{control} > 0
\end{cases}
$$

The mathematical elements are defined as follows:

$$
\begin{cases}
\mu_{treatment}: \text{Mean IRR for users in the Treatment arm} \\
\mu_{control}: \text{Mean IRR for users in the Control arm}
\end{cases}
$$


## **4. Statistical Evaluation Methodology**
### **4.1. Methodological Framework (Theory & Logic)**
#### **4.1.1. Primary Metric: Pairwise Proportion Testing & Bonferroni Correction**

Because the Control Arm is concurrently tested against two independent variants (ALS and LightGBM), the Family-Wise Error Rate (FWER) inflates. To maintain strict mathematical rigor, the Bonferroni Correction is applied. We divide the global significance level ($\alpha = 0.05$) by $m=2$ comparative pairs to secure the 95% confidence boundary.

Consequently, the adjusted significance threshold for the primary metric becomes $\alpha_{adj} = 0.025$. A treatment is only deemed statistically superior if its resulting P-value falls below this penalized boundary. The evaluation utilizes the two-sample Z-test for proportions, structured as:

$$
Z = \frac{\hat{p}_t - \hat{p}_c}{\sqrt{\hat{p}_{pool}(1 - \hat{p}_{pool}) \left(\frac{1}{n_t} + \frac{1}{n_c}\right)}}
$$

To satisfy C-level reporting requirements, the exact magnitude of the algorithmic uplift must be quantified using a Confidence Interval (CI) representing the delta ($\Delta$) between the proportions:

$$
CI_{\Delta} = (\hat{p}_t - \hat{p}_c) \pm Z_{1-\alpha_{adj}/2} \sqrt{\frac{\hat{p}_t(1-\hat{p}_t)}{n_t} + \frac{\hat{p}_c(1-\hat{p}_c)}{n_c}}
$$

*Note: A two-sided 95% Confidence Interval is reported alongside the one-tailed hypothesis test as standard practice for executive-facing reporting. The CI quantifies the plausible range of the uplift delta, while the one-tailed P-value evaluates whether that delta is directionally positive. Both are mathematically valid and mutually complementary.*

#### **4.1.2. Secondary Metric: Welch's T-Test and Effect Size**

The continuous secondary metric (Incremental Recommendation Revenue) violates the standard assumption of equal variances (homoscedasticity) required by a traditional Student’s T-Test. Fashion retail data is inherently heavily right-skewed, dominated by a small cohort of high-frequency "Whale" consumers who generate asymmetrical monetary density. To mathematically correct for unequal variances and differing sample sizes across the experimental arms, **Welch’s T-Test** is mandated:

$$t = \frac{\bar{X}_t - \bar{X}_c}{\sqrt{\frac{s_t^2}{n_t} + \frac{s_c^2}{n_c}}}$$

While the resulting P-value confirms statistical significance, it does not convey the magnitude of the economic shift. In an A/B test exhibiting unequal variances, pooling the standard deviations creates a mathematically biased effect size. Therefore, **Glass's Delta ($\Delta$)** is utilized instead of Cohen's $d$. Glass's Delta divides the mean difference strictly by the Control Group's standard deviation ($s_c$), anchoring the effect size to the stable baseline reference:

$$\Delta_{Glass} = \frac{\bar{X}_t - \bar{X}_c}{s_c}$$


In [4]:
# 1. Define Target Artifacts
artifacts = {
    "Test Ground Truth": "test_processed.parquet",
    "Article Master (for Prices)": "articles_processed.parquet",
    "ALS Predictions": "ALS_predictions.parquet",
    "LightGBM Features/Predictions": "lgbm_test_features.parquet",
    "Traffic Allocation": "ab_traffic_allocation.parquet"
}

print("--- ARTIFACT SCHEMA EXTRACTION ---")

# 2. Iterate and Extract Schemas
for name, filename in artifacts.items():
    file_path = DATA_DIR / filename
    if file_path.exists():
        # Lazy load to extract schema without RAM overhead
        lazy_df = pl.scan_parquet(file_path)
        schema = lazy_df.schema
        row_count = lazy_df.select(pl.len()).collect().item()
        
        print(f"\n[{name}] -> {filename}")
        print(f"Total Rows: {row_count:,}")
        print("Columns:")
        for col_name, col_type in schema.items():
            print(f"  - {col_name}: {col_type}")
    else:
        print(f"\n[{name}] -> {filename} (FILE NOT FOUND)")


--- ARTIFACT SCHEMA EXTRACTION ---

[Test Ground Truth] -> test_processed.parquet
Total Rows: 913,103
Columns:
  - t_dat: Date
  - price: Float64
  - sales_channel_id: Int64
  - quantity: UInt32
  - days_ago: Int64
  - time_weight: Float64
  - customer_id: Int32
  - article_id: Int32

[Article Master (for Prices)] -> articles_processed.parquet
Total Rows: 105,542
Columns:
  - product_code: Int64
  - prod_name: String
  - product_type_no: Int64
  - product_type_name: String
  - product_group_name: String
  - graphical_appearance_no: Int64
  - graphical_appearance_name: String
  - colour_group_code: Int64
  - colour_group_name: String
  - perceived_colour_value_id: Int64
  - perceived_colour_value_name: String
  - perceived_colour_master_id: Int64
  - perceived_colour_master_name: String
  - department_no: Int64
  - department_name: String
  - index_code: String
  - index_name: String
  - index_group_no: Int64
  - index_group_name: String
  - section_no: Int64
  - section_name: String
  

C:\Users\ASUS\AppData\Local\Temp\ipykernel_4428\2899654825.py:18: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  schema = lazy_df.schema


### **4.2. Pre-Execution Artifact Analysis**
#### **4.2.1. Data Architecture & Schema Analysis**

Prior to executing the statistical engine, the physical structure of the experimental artifacts must be validated to ensure deterministic relational joins. The schema extraction yields the following architectural footprint across the evaluation environment:

| Artifact Name | Dimensionality (Rows) | Primary Key(s) / Types | Target Metric / Value | Purpose in RCT Simulation | Architectural Status |
| :---: | :---: | :---: | :---: | :---: | :---: |
| `ab_traffic_allocation` | 216,479 | `customer_id` (Int32) | `experimental_arm` (String) | The master routing table. Determines which algorithmic output is evaluated for each user. | **Ready** |
| `test_processed` | 913,103 | `customer_id` (Int32), `article_id` (Int32) | `price` (Float64) | Ground truth interactions. Used to determine hits (Primary) and calculate IRR (Secondary). | **Ready** |
| `ALS_predictions` | 2,597,748 | `customer_id` (Int64), `article_id` (Int64) | `rank` (Int64) | Static recommendation vectors for Treatment A ($216,479 \times 12$). Note the Int64 casting discrepancy. | **Requires Int32 Downcast** |
| `lgbm_test_features` | 17,054,516 | `customer_id` (Int32), `article_id` (Int32) | `label` (Int8), `features` | Unscored candidate pairs for Treatment B. Lacks explicit LightGBM probability scores or Top-K ranks. | **Deprecated** (Replaced by `lgbm_predictions`) |
| `articles_processed` | 105,542 | `article_id` (Int32) | `index_name`, etc. | Master catalog. Reserved for metadata fallbacks or ECR computations if required. | **Ready** |
| `MBA_predictions` | - | `customer_id` (Int32), `article_id` (Int32) | `rank` (Int8) | Static, pre-computed recommendation vectors for Control Arm. | **Currently not generated in Phase 2** |
| `lgbm_predictions` | - | `customer_id` (Int32), `article_id` (Int32) | `rank` (Int8) | Static, pre-computed recommendation vectors for Treatment B. | **Currently not generated in Phase 2** |

#### **4.2.2. Architectural Normalization & Artifact Export**

An objective analysis of the extracted schemas reveals three critical path dependencies that must be resolved prior to calculating the T-scores and Z-scores:

1.  **Data Type Misalignment in ALS Artifact:** The `ALS_predictions.parquet` file utilizes `Int64` for both `customer_id` and `article_id`, whereas the master allocation table and ground truth utilize `Int32`. A strict downcast operation (`pl.col().cast(pl.Int32)`) must be applied during ingestion to prevent memory bloat and failed equi-joins.

2.  **Missing LightGBM Rank/Score:** The extracted `lgbm_test_features.parquet` contains 17.05 million rows of raw user-item feature pairs and binary labels, but it **lacks a LightGBM prediction score or top-K rank**. Evaluating Treatment B requires an ordered list of exactly $K=12$ recommendations per user. 

3.  **Dynamic MBA Baseline Computation:** The Control arm (MBA FP-Growth) relies on dynamic generation via `Association_Rules.parquet`. Executing dynamic rule evaluation for the 73,586 Control users concurrently within the statistical loop will introduce severe RAM constraints and processing bottlenecks. 

$\implies$ **Standardization Protocol:** To guarantee flawless execution of the A/B test simulation, the predictive outputs of all three experimental arms must be standardized into an identical, highly compressed static schema: 
`[customer_id (Int32), article_id (Int32), rank (Int8)]` 
where `rank` strictly bounded $\in [1, 12]$.

$\implies$ Add new code blocks to export `MBA_predictions.parquet` and `lgbm_predictions.parquet` from the Phase 2 notebooks

### **4.3. Simulated RCT Execution & Statistical Inference**

To definitively evaluate the algorithmic uplift, the statistical engine will fuse the master assignment router, the unified predictive artifacts, and the historical ground truth. 

**Data Fusion & Evaluation Logic**

1.  **Isolate & Map:** The prediction artifacts (`MBA`, `ALS`, `LightGBM`) are strictly filtered to contain only the users assigned to their respective experimental arms (Control, Treatment A, Treatment B).
2.  **Ground Truth Intersection:** The unified prediction matrix is joined against `test_processed.parquet` using `[customer_id, article_id]` as the composite key. A successful intersection represents an empirical "Hit."
3.  **User-Level Aggregation:** 
    *   **Hit Rate Propensity (Binary):** A user is assigned a `1` if the intersection yields $\ge 1$ item, otherwise `0`.
    *   **Price-Weighted Hit Score (Continuous):** The `price` column from the ground truth is summed for all intersecting items per user. Non-converting users receive a strict `0.0`.

**Mathematical Formulation (Programmatic Implementation)**

1. **Primary Metric (Proportion Z-Test):**

To adhere to the Bonferroni Correction ($\alpha_{adj} = 0.025$), the critical Z-value for the 95% Confidence Interval is calculated via the inverse Cumulative Distribution Function (CDF): 

$$Z_{crit} = \Phi^{-1}(1 - \alpha_{adj}/2) \approx 2.2414$$

The Confidence Interval for the delta ($\Delta = \hat{p}_{treatment} - \hat{p}_{control}$) is computed using the unpooled standard error:

$$SE_{\Delta} = \sqrt{\frac{\hat{p}_t(1-\hat{p}_t)}{n_t} + \frac{\hat{p}_c(1-\hat{p}_c)}{n_c}}$$

$$CI = \Delta \pm (Z_{crit} \times SE_{\Delta})$$

2. **Secondary Metric (Welch's T-Test & Cohen's d):**
The continuous arrays are evaluated using `scipy.stats.ttest_ind(equal_var=False, alternative='greater')`. To measure standard deviation shifts independent of sample size, **Cohen's $d$** is computed:

$$d = \frac{\bar{X}_t - \bar{X}_c}{\sqrt{\frac{s_t^2 + s_c^2}{2}}}$$

#### **4.3.1. Statistical Engine Implementation**

In [5]:
import polars as pl
import numpy as np
import scipy.stats as stats
from pathlib import Path

# --- 1. Path Resolution ---
cwd = Path.cwd()
if (cwd / "data" / "processed").exists():
    DATA_DIR = cwd / "data" / "processed"
else:
    DATA_DIR = cwd.parent / "data" / "processed"

print("Building fully optimized Polars Lazy execution graph...")

# --- 2. Load ALL Artifacts as LazyFrames ---
alloc_lazy = pl.scan_parquet(DATA_DIR / "ab_traffic_allocation.parquet")

# FIX: Explicit subset deduplication to prevent price double-counting
gt_lazy = (
    pl.scan_parquet(DATA_DIR / "test_processed.parquet")
    .select(['customer_id', 'article_id', 'price'])
    .unique(subset=['customer_id', 'article_id'], keep='last')
)

mba_lazy = pl.scan_parquet(DATA_DIR / "MBA_predictions.parquet")
als_lazy = pl.scan_parquet(DATA_DIR / "ALS_predictions.parquet")
lgbm_lazy = pl.scan_parquet(DATA_DIR / "lgbm_predictions.parquet")

# --- 3. Filter Predictions by Assigned Arms ---
ctrl_users = alloc_lazy.filter(pl.col("experimental_arm") == "Control").select("customer_id")
trt_a_users = alloc_lazy.filter(pl.col("experimental_arm") == "Treatment_A").select("customer_id")
trt_b_users = alloc_lazy.filter(pl.col("experimental_arm") == "Treatment_B").select("customer_id")

# Construct Join Logic (ALS downcast happens here)
mba_preds = (
    mba_lazy.join(ctrl_users, on="customer_id", how="inner")
    .select(['customer_id', 'article_id'])
    .with_columns(pl.lit("Control").alias("arm"))
)

als_preds = (
    als_lazy
    .with_columns([pl.col("customer_id").cast(pl.Int32), pl.col("article_id").cast(pl.Int32)])
    .join(trt_a_users, on="customer_id", how="inner")
    .select(['customer_id', 'article_id'])
    .with_columns(pl.lit("Treatment_A").alias("arm"))
)

lgbm_preds = (
    lgbm_lazy.join(trt_b_users, on="customer_id", how="inner")
    .select(['customer_id', 'article_id'])
    .with_columns(pl.lit("Treatment_B").alias("arm"))
)

all_preds = pl.concat([mba_preds, als_preds, lgbm_preds])

# --- 4. Ground Truth Intersection & Aggregation ---
hits_lazy = all_preds.join(gt_lazy, on=['customer_id', 'article_id'], how="inner")

# FIX: Version-safe aggregation for is_hit
user_metrics_lazy = (
    alloc_lazy.join(
        hits_lazy.group_by("customer_id").agg([
            pl.len().cast(pl.Boolean).cast(pl.Int8).alias("is_hit"),
            pl.col("price").sum().alias("irr")
        ]),
        on="customer_id", how="left"
    )
    .with_columns([
        pl.col("is_hit").fill_null(0),
        pl.col("irr").fill_null(0.0)
    ])
)

# --- 5. Execute Graph (Materialize to RAM) ---
print("Executing graph and materializing results...")
user_metrics_pd = user_metrics_lazy.collect().to_pandas()

# --- 6. Statistical Execution ---
print("\n" + "="*70)
print("SECTION 4: FORMAL STATISTICAL EVALUATION RESULTS")
print("="*70)

alpha_adj = 0.05 / 2 # Bonferroni Correction
z_crit = stats.norm.ppf(1 - alpha_adj/2)

ctrl_data = user_metrics_pd[user_metrics_pd['experimental_arm'] == 'Control']

for trt_name in ['Treatment_A', 'Treatment_B']:
    trt_data = user_metrics_pd[user_metrics_pd['experimental_arm'] == trt_name]
    
    # Vectors
    c_hit, t_hit = ctrl_data['is_hit'].values, trt_data['is_hit'].values
    c_irr, t_irr = ctrl_data['irr'].values, trt_data['irr'].values
    nc, nt = len(c_hit), len(t_hit)
    
    # Primary Metric: Proportion Test
    pc, pt = c_hit.mean(), t_hit.mean()
    p_pool = (c_hit.sum() + t_hit.sum()) / (nc + nt)
    se_pool = np.sqrt(p_pool * (1 - p_pool) * (1/nc + 1/nt))
    z_stat = (pt - pc) / se_pool
    p_val_z = stats.norm.sf(z_stat)
    
    delta = pt - pc
    se_unpooled = np.sqrt((pt*(1-pt)/nt) + (pc*(1-pc)/nc))
    ci_lower = delta - (z_crit * se_unpooled)
    ci_upper = delta + (z_crit * se_unpooled)
    
    # Secondary Metric: Welch's T-Test
    t_stat, p_val_t = stats.ttest_ind(t_irr, c_irr, equal_var=False, alternative='greater')
    
    # FIX: Glass's Delta (uses Control standard deviation due to unequal variances)
    glass_delta = (np.mean(t_irr) - np.mean(c_irr)) / np.std(c_irr, ddof=1)
    
    model_name = "ALS (Collaborative Filtering)" if trt_name == 'Treatment_A' else "LightGBM (Learning-to-Rank)"
    print(f"\n[ EXPERIMENTAL ARM: {trt_name} | {model_name} ]")
    print("-" * 70)
    print("PRIMARY METRIC: Hit Rate Propensity (Binary)")
    print(f"  Control Hit Rate   : {pc:.4%} (n={nc:,})")
    print(f"  Treatment Hit Rate : {pt:.4%} (n={nt:,})")
    print(f"  Absolute Uplift (Δ): {delta:.4%} | Relative Uplift: {(pt-pc)/pc:.2%}")
    print(f"  Z-Score            : {z_stat:.4f}")
    print(f"  P-Value (1-tail)   : {p_val_z:.2e}")
    print(f"  95% CI for Δ       : [{ci_lower:.4%}, {ci_upper:.4%}]")
    sig_z = "SIGNIFICANT" if p_val_z < alpha_adj else "NOT SIGNIFICANT"
    print(f"  Result (α={alpha_adj:.3f}) : {sig_z}")
    
    print("\nSECONDARY METRIC: Incremental Recommendation Revenue (IRR)")
    print(f"  Control Mean IRR   : {np.mean(c_irr):.6f} | Std: {np.std(c_irr):.6f}")
    print(f"  Treatment Mean IRR : {np.mean(t_irr):.6f} | Std: {np.std(t_irr):.6f}")
    print(f"  Absolute IRR Shift : {np.mean(t_irr) - np.mean(c_irr):.6f}")
    print(f"  Welch's T-Score    : {t_stat:.4f}")
    print(f"  P-Value (1-tail)   : {p_val_t:.2e}")
    print(f"  Glass's Delta      : {glass_delta:.4f}")
    sig_t = "SIGNIFICANT" if p_val_t < alpha_adj else "NOT SIGNIFICANT"
    print(f"  Result (α={alpha_adj:.3f}) : {sig_t}")

# --- 7. SEGMENT STRATIFICATION FOR SECTION 5 ---
print("\n" + "="*70)
print("SECTION 5.2: SEGMENT-STRATIFIED A/B RESULTS")
print("="*70)

# FIX: Use the correct column name "segment_label" (matching Phase 1 schema)
segments_lazy = pl.scan_parquet(DATA_DIR / "customers_segmented.parquet").select(["customer_id", "segment_label"])
user_metrics_pl = pl.from_pandas(user_metrics_pd)

user_metrics_segmented = (
    user_metrics_pl
    .lazy()
    .join(segments_lazy, on="customer_id", how="left")
    .collect()
)

segment_breakdown = (
    user_metrics_segmented
    .group_by(["segment_label", "experimental_arm"])
    .agg([
        pl.col("is_hit").mean().alias("hit_rate"),
        pl.col("irr").mean().alias("mean_irr"),
        pl.len().alias("n_users")
    ])
    .with_columns((pl.col("hit_rate") * 100).round(4).alias("hit_rate_pct"))
    .sort(["segment_label", "experimental_arm"])
)

pl.Config.set_tbl_rows(20)
print(segment_breakdown)

Building fully optimized Polars Lazy execution graph...
Executing graph and materializing results...



SECTION 4: FORMAL STATISTICAL EVALUATION RESULTS

[ EXPERIMENTAL ARM: Treatment_A | ALS (Collaborative Filtering) ]
----------------------------------------------------------------------
PRIMARY METRIC: Hit Rate Propensity (Binary)
  Control Hit Rate   : 4.3881% (n=73,586)
  Treatment Hit Rate : 4.7989% (n=71,496)
  Absolute Uplift (Δ): 0.4108% | Relative Uplift: 9.36%
  Z-Score            : 3.7380
  P-Value (1-tail)   : 9.27e-05
  95% CI for Δ       : [0.1643%, 0.6573%]
  Result (α=0.025) : SIGNIFICANT

SECONDARY METRIC: Incremental Recommendation Revenue (IRR)
  Control Mean IRR   : 0.001360 | Std: 0.007174
  Treatment Mean IRR : 0.001439 | Std: 0.007897
  Absolute IRR Shift : 0.000079
  Welch's T-Score    : 1.9874
  P-Value (1-tail)   : 2.34e-02
  Glass's Delta      : 0.0110
  Result (α=0.025) : SIGNIFICANT

[ EXPERIMENTAL ARM: Treatment_B | LightGBM (Learning-to-Rank) ]
----------------------------------------------------------------------
PRIMARY METRIC: Hit Rate Propensity (Bina

#### **4.3.2. Empirical Outcome Analysis**

The statistical engine successfully evaluated the simulated interactions of 216,479 unique test subjects against the historical ground truth. To ensure maximum analytical clarity, the empirical outputs are aggregated into three distinct evaluation matrices: separating the probability of conversion, the magnitude of economic generation, and the stratified behavioral variance across user clusters.

##### **4.3.2.1. Statistical Aggregation Matrices**

**Table 1: Primary Metric – Hit Rate Propensity (Binary Conversion)**

*Evaluation Threshold: $\alpha_{adj} = 0.025$ (One-Tailed Z-Test)*

| Experimental Arm | Sample Size ($N$) | Hit Rate | Absolute Uplift ($\Delta$) | Relative Uplift | Z-Score | P-Value | 95% Confidence Interval | Result |
| :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **Control** (MBA) | 73,586 | 4.3881% | — | — | — | — | — | Baseline |
| **Treatment A** (ALS) | 71,496 | 4.7989% | +0.4108% | +9.36% | 3.7380 | $9.27 \times 10^{-5}$ | $[0.1643\%, 0.6573\%]$ | **Significant** |
| **Treatment B** (LGBM) | 71,397 | 7.5073% | +3.1193% | +71.09% | 25.1523 | $6.66 \times 10^{-140}$ | $[2.8409\%, 3.3976\%]$ | **Significant** |

**Table 2: Secondary Metric – Incremental Recommendation Revenue (IRR)**

*Evaluation Threshold: $\alpha_{adj} = 0.025$ (Welch's T-Test)*

| Experimental Arm | Mean IRR | Std. Deviation | Absolute Shift | Welch's T-Score | P-Value | Glass's $\Delta$ | Result |
| :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **Control** (MBA) | 0.001360 | 0.007174 | — | — | — | — | Baseline |
| **Treatment A** (ALS) | 0.001439 | 0.007897 | +0.000079 | 1.9874 | $2.34 \times 10^{-2}$ | 0.0110 | **Significant** |
| **Treatment B** (LGBM) | 0.002671 | 0.010267 | +0.001312 | 28.1194 | $9.66 \times 10^{-174}$| 0.1828 | **Significant** |

**Table 3: Segment-Stratified Empirical Hit Rates**

*Analysis of variance across pre-defined RFM clusters.*

| Segment Label | Control Hit Rate | Treatment A (ALS) | ALS Relative Δ | Treatment B (LGBM) | LGBM Relative Δ |
| :---: | :---: | :---: | :---: | :---: | :---: |
| **Mature Champions** | 4.3359% | 5.0625% | **+16.76%** | 7.9483% | **+83.31%** |
| **Mature Newcomers** | 4.7848% | 3.8701% | **−19.11%** | 6.5616% | **+37.13%** |
| **Young At-Risk** | 4.4572% | 2.9616% | **−33.55%** | 6.1375% | **+37.69%** |
| **Young Core** | 4.2204% | 4.6514% | **+10.21%** | 6.4903% | **+53.78%** |
| **Young Whales** | 4.4231% | 4.9566% | **+12.06%** | 7.7705% | **+75.68%** |


##### **4.3.2.2. Analytical Interpretations & Statistical Implications**

**1. The Treatment A (ALS) Paradigm: Quantifying the "ROI Trap"**

The empirical data from Treatment A perfectly encapsulates the **"ROI Trap"** - a scenario where a statistical test yields mathematical significance but fails to deliver material economic utility. For the **Primary Metric**, the **Collaborative Filtering** architecture successfully utilized **Latent Factorization** to drive a **Relative Uplift** of 9.36% in conversion propensity, achieving a definitive **Z-Score** of 3.7380. 

Critically, following deduplication normalization, ALS also managed to clear the **Bonferroni-corrected threshold** ($\alpha_{adj} = 0.025$) for the **Secondary Metric**, returning a **Welch's T-Test P-Value** of $0.0234$. However, despite achieving formal statistical significance across both axes, the economic magnitude is practically negligible. The **Absolute IRR Shift** (+0.000079) equates to a microscopic **Glass's Delta** ($\Delta = 0.0110$). 

Unlike the LightGBM model which utilized **Expected Cross-Sell Revenue (ECR)** pricing thresholds and **Wallet-Matching**, the pure Matrix Factorization of ALS blindly optimized for structural co-occurrence. This made it fatally susceptible to **Popularity Bias**. It marginally increased the *frequency* of purchases (Hit Rate), but cannibalized margin density by recommending cheaper, high-velocity basics, proving that mathematical accuracy in a latent space does not natively guarantee increased basket scale.

**2. Simpson's Paradox and Empirical Segment Harm**

An examination of **Table 3** reveals a critical systemic flaw within the ALS architecture. The aggregate +9.36% Hit Rate uplift reported in Table 1 is mathematically misleading - it is an artifact of **Simpson's Paradox**. 

While ALS generated positive lifts for high-frequency segments like **Mature Champions** (+16.76%) and **Young Whales** (+12.06%), it actively destroyed value for dormant users. **Young At-Risk** users subjected to ALS converted at a rate **33.55% lower** than the MBA Control, and **Mature Newcomers** dropped by **19.11%**. Mechanically, Collaborative Filtering fails on lapsed/new users because their sparse interaction history is insufficient to anchor meaningful latent vectors. Consequently, ALS feeds them generic popularity lists that perform markedly worse than the demographically-aware **Association Rules** utilized by the Control arm.

**3. The Treatment B (LightGBM) Dominance: Universal Optimization**

The performance of Treatment B establishes **Gradient Boosting** as the unequivocally superior algorithmic architecture. On the **Primary Metric**, LightGBM achieved an astronomical **Z-Score** of 25.1523, equating to a **P-value** approaching absolute zero ($6.66 \times 10^{-140}$). The algorithm increased the conversion probability by a staggering **Relative Uplift** of 71.09%, with a tightly constrained **95% Confidence Interval** that operates entirely above a 2.8% absolute gain.

Crucially, LightGBM successfully cleared the dual-metric optimization hurdle while demonstrating universal segment superiority. The **Secondary Metric** demonstrates that the **Incremental Recommendation Revenue (IRR)** effectively doubled (from 0.001360 to 0.002671). The **Welch's T-Score** of 28.1194 mathematically confirms this systemic shift. The calculated **Glass's Delta** ($\Delta = 0.1828$) represents a highly tangible, operationally impactful effect size, proving the revenue shift scales robustly relative to the control group's baseline variance. 

**4. Validation of the Anti-Leakage Methodology**

These statistical milestones must be contextualized within the **Anti-Join Methodology** enforced during Phase 2. Because trivial repurchases were strictly barred from the candidate sets, the calculated **Uplifts** represent 100% pure **Novel Discovery**. The evaluation proves that advanced, multi-dimensional feature matrices possess the operational capacity to stimulate genuine, previously dormant consumer demand without relying on historical repurchasing crutches.


## **5. Business Impact & Deployment Decision**

### **5.1 ROI Trap Analysis & Annualized Revenue Projection**

The primary function of a Data-Driven Marketing strategy is to maximize the **Return on Investment (ROI)**. The **"ROI Trap"** occurs when an organization deploys a statistically significant algorithm whose computational infrastructure costs vastly exceed its marginal revenue generation. To make an objective deployment decision, the empirical **Absolute IRR Shift** calculated in Section 4 must be projected into an annualized economic forecast.

The simulation evaluated an active cohort of $216,479$ users over a strictly bounded 28-day evaluation window. By utilizing the total segmented customer base ($1,338,785$ users) mapped during the RFM classification phase, the **Annualized Incremental Revenue** is forecasted utilizing the following scaling formula:

$$
\text{Projected Annual Incremental Revenue} = \left( \Delta_{\mu} \times N_{Total} \right) \times \left( \frac{365}{28} \right)
$$

Where elements are defined as:

*   $\Delta_{\mu}$: **Absolute IRR Shift** (Continuous Incremental Recommendation Revenue per user).
*   $N_{Total}$: **Total Addressable Market** ($1.33$ million active users).
*   $365 / 28$: **Temporal Annualization Multiplier** ($\approx 13.035$).

**Table 4: Annualized Economic Projection**

| Algorithmic Architecture | Absolute IRR Shift ($\Delta_{\mu}$) | Statistical Validity | Projected Annual Incremental Yield | Computational Inference Cost |
| :---: | :---: | :---: | :---: | :---: |
| **ALS (Collaborative Filtering)** | +0.000079 | **Passed** ($p < \alpha_{adj}$) | +1,379 NRU | Medium (Latent Matrix Multiplication) |
| **LightGBM (Learning-to-Rank)** | +0.001312 | **Passed** ($p \ll \alpha_{adj}$) | +22,897 NRU | High (Multi-Dimensional Feature Trees) |

*Note: Monetary values are expressed in Normalized Revenue Units (NRU), corresponding to the normalized price scale [0, 0.591] used in the retail dataset. Conversion to absolute currency requires the original dataset normalization scalar. For executive reporting, the directional ratio (LightGBM generating ~16.6× the yield of ALS) remains economically valid regardless of unit scale.*

**The Inference Cost Anchor:**

To complete the ROI evaluation, the projected annual incremental yield of +22,897 NRU must be weighed against LightGBM's inference costs. In production, this is measured in cloud compute units per batch scoring run (e.g., GPU/CPU-hour pricing per million candidate pairs). The scale of the yield differential—22,897 NRU for LightGBM versus ALS's negligible 1,379 NRU—provides a massive economic margin of safety. Deployment is strictly justified provided that monthly inference infrastructure costs remain below the annualized yield divided by 12 ($\approx 1,908$ NRU/month). This threshold must be validated against live cloud billing data prior to enterprise-wide scaling.


### **5.2 Segment-Stratified Interpretation (RFM Lens)**

To understand *why* LightGBM overwhelmingly succeeded where ALS triggered the ROI Trap, the empirical metrics must be viewed through the RFM Cluster definitions established in Phase 1. 

**The ALS Simpson's Paradox:**

The aggregate Hit Rate uplift (+9.36%) produced by ALS masks severe structural failures. As demonstrated in the Segment Stratification matrix, ALS's aggregate positive performance is entirely driven by high-frequency segments (Mature Champions: +16.76%, Young Whales: +12.06%). Shockingly, ALS actively harmed dormant segments: **Young At-Risk** users converted at a rate **33.55% lower** than the Control arm, and **Mature Newcomers** dropped by **19.11%**. 

This phenomenon is caused by **Popularity Bias**. Because Collaborative Filtering treats all interactions uniformly within a latent matrix, it struggles with cold or lapsed users whose sparse history cannot anchor meaningful vectors. Consequently, ALS feeds them generic, cheap basics that fail to convert. It blindly optimizes for structural co-occurrence, entirely missing the margin constraints of the consumer.

**The LightGBM Prescriptive Personalization:**

Conversely, LightGBM generated positive uplifts in *every single segment*, effectively solving the sparse-history deficit. By incorporating **Wallet Matching** (`user_avg_price`) and **Expected Cross-Sell Revenue (ECR)** thresholds directly into the decision trees, the ranker actively optimized price acceptance:

*   **The "Whale" Optimization:** For **Young Whales** (who represent 54.35% of total revenue), LightGBM recognized their historically high `user_avg_price` and algorithmically filtered out low-margin basics, driving an astonishing 75.68% Hit Rate uplift while pushing the mean IRR to 0.002836.
*   **The "At-Risk" Reactivation:** For **Young At-Risk** consumers, LightGBM successfully avoided "sticker shock" by downgrading premium candidates, routing demographically appropriate items that secured a +37.69% Hit Rate reversal over the Control baseline.

### **5.3 C-Level Prescriptive Strategy & Deployment Mandate**

Based on the statistical, economic, and segment-stratified evidence compiled throughout this Randomized Controlled Trial, the following strategic mandates are issued:

**1. Deprecate ALS; Transition to Supervised Learning-to-Rank**

The organization must immediately deprecate purely implicit matrix factorization (ALS) as a primary ranking engine. ALS successfully triggers interaction density but actively harms margin density and misfires on sparse-history users. The retail catalog exhibits a 90-day shelf life, mandating the use of Supervised Learning-to-Rank architectures capable of utilizing dynamic, multi-dimensional pricing and demographic features.

**2. Enforce "Novel Discovery" Metrics System-Wide**

The success of the LightGBM model was predicated on an **Anti-Join Architecture** that strictly blocked trivial repurchases during candidate generation. Marketing KPIs must reflect this paradigm shift. Replace Gross Hit Rate with **Net-New Hit Rate**, and substitute standard AOV with **Incremental Recommendation Revenue (IRR)**. These metrics measure true demand generation rather than artificially inflated repurchase tracking.

**3. Deploy Dual-Engine Prescriptive Logic**

The final production environment will utilize a bipartite algorithmic approach tailored to computational latency constraints:

*   **Primary Navigation & Homepage (LightGBM):** Route traffic through the LightGBM L2R engine to govern the largest IRR opportunity. **Execution Protocol:** Initiate a **20% Phased Rollout**. Allocate 20% of live homepage traffic to LightGBM while monitoring real-time guardrail metrics (page latency, cart-to-checkout drop rate, and app crash rate) for a minimum of 14 days (capturing two full weekly seasonality cycles). Execute a 100% rollout only upon guardrail clearance.

*   **Shopping Cart & Checkout (MBA Baseline):** Retain the MBA FP-Growth Baseline utilizing the Expected Cross-Sell Revenue (ECR) heuristic. Because LightGBM requires heavy batch feature engineering, it is ideal for asynchronous Homepage generation. Conversely, MBA Association Rules (N-to-1) execute via instant $O(1)$ dictionary lookups, making them the computationally optimal choice for real-time, impulse-buy generation at checkout.

In [6]:
import os
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path

# =====================================================================
# 1. Robust Path Resolution
# =====================================================================
cwd = Path.cwd()
if (cwd / "data" / "processed").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data" / "processed").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not locate project root.")

# Target the top-level figures folder
output_dir = PROJECT_ROOT / "figures" / "06_abtest"
output_dir.mkdir(parents=True, exist_ok=True)

# =====================================================================
# FIGURE 2: Segment-Stratified Hit Rates (Simpson's Paradox)
# =====================================================================
segments = ['Mature Champions', 'Mature Newcomers', 'Young At-Risk', 'Young Core', 'Young Whales']

# Hit Rate Data
hr_control = [4.34, 4.78, 4.46, 4.22, 4.42]
hr_als     = [5.06, 3.87, 2.96, 4.65, 4.96]
hr_lgbm    = [7.95, 6.56, 6.14, 6.49, 7.77]

# Find absolute max and min for highlighting
all_values = hr_control + hr_als + hr_lgbm
max_val = max(all_values)
min_val = min(all_values)

def format_label(val):
    """Bolds only the global maximum and global minimum values."""
    if val == max_val or val == min_val:
        return f"<b>{val}%</b>"
    return f"{val}%"

# Economy-focused Corporate Palette
color_control = '#7F8C8D' # Slate Gray (Baseline)
color_als     = '#2980B9' # Steel Blue (Alternative 1)
color_lgbm    = '#27AE60' # Emerald Green (Winning/Profit Alternative)

# Layout configuration
layout_defaults = dict(
    plot_bgcolor='white',
    font=dict(family="Roboto, 'Segoe UI', sans-serif", color="#333333"),
    # Y-axis title smaller (size 10), Y-axis ticks larger (size 13), Gridlines REMOVED
    yaxis=dict(
        showgrid=False, 
        zeroline=True, zerolinecolor='lightgray',
        title=dict(text="Hit Rate Propensity (%)", font=dict(size=14)), 
        tickfont=dict(size=13)
    ),
    # X-axis ticks larger (size 13)
    xaxis=dict(
        showgrid=False, 
        tickfont=dict(size=13), 
        linecolor='lightgray'
    ),
    legend=dict(yanchor="middle", y=0.5, xanchor="left", x=1.02, font=dict(size=12), bordercolor="lightgray", borderwidth=1),
    margin=dict(t=80, b=50, l=60, r=180)
)

fig = go.Figure()

# Add Control Trace
fig.add_trace(go.Bar(
    x=segments, y=hr_control, 
    name='Control (MBA Baseline)', 
    marker_color=color_control,
    text=[format_label(v) for v in hr_control], 
    textposition='outside', textfont=dict(size=11)
))

# Add ALS Trace
fig.add_trace(go.Bar(
    x=segments, y=hr_als, 
    name='Treatment A (ALS)', 
    marker_color=color_als,
    text=[format_label(v) for v in hr_als], 
    textposition='outside', textfont=dict(size=11)
))

# Add LGBM Trace
fig.add_trace(go.Bar(
    x=segments, y=hr_lgbm, 
    name='Treatment B (LightGBM)', 
    marker_color=color_lgbm,
    text=[format_label(v) for v in hr_lgbm], 
    textposition='outside', textfont=dict(size=11)
))

# Add Title & Subtitle
fig.update_layout(
    **layout_defaults,
    title=dict(
        text="<b>Simpson's Paradox: Algorithmic Performance Across RFM Segments</b><br>"
             "<span style='font-size: 13px; color: #666666'>"
             "LightGBM demonstrates universal segment optimization, while ALS exhibits significant performance inversion in sparse-history cohorts.</span>",
        font=dict(size=18),
        y=0.92
    ),
    barmode='group'
)
fig.update_yaxes(range=[0, 9.0]) # Provide headroom for data labels

# Save and Display
output_path = output_dir / "simpsons_paradox_segments.png"
fig.write_image(output_path, width=950, height=500, scale=2)
fig.show()

print(f"Successfully generated and saved figure to:\n{output_path}")

Successfully generated and saved figure to:
d:\_Python\Year3_Semester2\DDM\final_group_project\hm-recommendation-system\figures\06_abtest\simpsons_paradox_segments.png


In [7]:
# %%
# =====================================================================
# POWERBI EXPORT: A/B Test Summary Tables for Dashboard (Tab 3)
# =====================================================================
import polars as pl
import numpy as np
import scipy.stats as stats

print("Exporting A/B Test summary artifacts for PowerBI...")

# --- Rebuild user_metrics if kernel was restarted, otherwise reuse ---
# (If user_metrics_pd is still in memory from Section 4, skip the reload block)
try:
    _ = user_metrics_pd
    print("  Reusing user_metrics_pd from Section 4 execution.")
except NameError:
    print("  user_metrics_pd not found in memory. Rebuilding from parquets...")
    alloc = pl.read_parquet(DATA_DIR / "ab_traffic_allocation.parquet")
    gt = (pl.scan_parquet(DATA_DIR / "test_processed.parquet")
          .select(['customer_id','article_id','price'])
          .unique(subset=['customer_id','article_id'], keep='last').collect())
    mba = pl.read_parquet(DATA_DIR / "MBA_predictions.parquet")
    als = pl.read_parquet(DATA_DIR / "ALS_predictions.parquet").with_columns(
        [pl.col("customer_id").cast(pl.Int32), pl.col("article_id").cast(pl.Int32)])
    lgbm = pl.read_parquet(DATA_DIR / "lgbm_predictions.parquet")
    ctrl_ids  = alloc.filter(pl.col("experimental_arm")=="Control")["customer_id"]
    trta_ids  = alloc.filter(pl.col("experimental_arm")=="Treatment_A")["customer_id"]
    trtb_ids  = alloc.filter(pl.col("experimental_arm")=="Treatment_B")["customer_id"]
    mba_p  = mba.filter(pl.col("customer_id").is_in(ctrl_ids)).select(["customer_id","article_id"])
    als_p  = als.filter(pl.col("customer_id").is_in(trta_ids)).select(["customer_id","article_id"])
    lgbm_p = lgbm.filter(pl.col("customer_id").is_in(trtb_ids)).select(["customer_id","article_id"])
    all_preds = pl.concat([mba_p, als_p, lgbm_p])
    hits = all_preds.join(gt, on=["customer_id","article_id"], how="inner")
    hit_agg = hits.group_by("customer_id").agg([
        pl.len().cast(pl.Boolean).cast(pl.Int8).alias("is_hit"),
        pl.col("price").sum().alias("irr")
    ])
    user_metrics_pd = (alloc.join(hit_agg, on="customer_id", how="left")
                       .with_columns([pl.col("is_hit").fill_null(0), pl.col("irr").fill_null(0.0)])
                       .to_pandas())

# --- 1. ARM-LEVEL SUMMARY TABLE ---
alpha_adj = 0.025
arm_rows = []
ctrl = user_metrics_pd[user_metrics_pd["experimental_arm"] == "Control"]

for arm, model in [("Treatment_A", "ALS"), ("Treatment_B", "LightGBM")]:
    trt = user_metrics_pd[user_metrics_pd["experimental_arm"] == arm]
    pc, pt = ctrl["is_hit"].mean(), trt["is_hit"].mean()
    nc, nt = len(ctrl), len(trt)
    p_pool = (ctrl["is_hit"].sum() + trt["is_hit"].sum()) / (nc + nt)
    se_pool = np.sqrt(p_pool*(1-p_pool)*(1/nc+1/nt))
    z_stat = (pt - pc) / se_pool
    p_val_z = stats.norm.sf(z_stat)
    t_stat, p_val_t = stats.ttest_ind(trt["irr"], ctrl["irr"], equal_var=False, alternative="greater")
    glass_d = (trt["irr"].mean() - ctrl["irr"].mean()) / np.std(ctrl["irr"], ddof=1)
    arm_rows.append({
        "arm": arm, "model": model,
        "n_users": nt,
        "hit_rate": round(pt * 100, 4),
        "hit_rate_control": round(pc * 100, 4),
        "abs_uplift_pct": round((pt - pc) * 100, 4),
        "rel_uplift_pct": round((pt - pc) / pc * 100, 2),
        "z_score": round(z_stat, 4),
        "p_value_z": float(f"{p_val_z:.4e}"),
        "significant_primary": p_val_z < alpha_adj,
        "mean_irr": round(trt["irr"].mean(), 6),
        "mean_irr_control": round(ctrl["irr"].mean(), 6),
        "irr_shift": round(trt["irr"].mean() - ctrl["irr"].mean(), 6),
        "welch_t": round(t_stat, 4),
        "p_value_t": float(f"{p_val_t:.4e}"),
        "glass_delta": round(glass_d, 4),
        "significant_secondary": p_val_t < alpha_adj,
        "annual_nru": round((trt["irr"].mean() - ctrl["irr"].mean()) * 1_338_785 * (365/28), 0),
    })

# Add control row for completeness
arm_rows.insert(0, {
    "arm": "Control", "model": "MBA FP-Growth",
    "n_users": len(ctrl),
    "hit_rate": round(ctrl["is_hit"].mean()*100, 4),
    "hit_rate_control": None, "abs_uplift_pct": 0.0, "rel_uplift_pct": 0.0,
    "z_score": 0.0, "p_value_z": 1.0, "significant_primary": False,
    "mean_irr": round(ctrl["irr"].mean(), 6), "mean_irr_control": None,
    "irr_shift": 0.0, "welch_t": 0.0, "p_value_t": 1.0,
    "glass_delta": 0.0, "significant_secondary": False, "annual_nru": 0.0,
})

ab_arm_summary = pl.DataFrame(arm_rows)

# --- 2. SEGMENT-LEVEL BREAKDOWN TABLE ---
segments = pl.read_parquet(DATA_DIR / "customers_segmented.parquet").select(["customer_id", "segment_label"])
um_pl = pl.from_pandas(user_metrics_pd).join(segments, on="customer_id", how="left")

seg_breakdown = (
    um_pl.group_by(["segment_label", "experimental_arm"])
    .agg([
        pl.col("is_hit").mean().alias("hit_rate_pct"),
        pl.col("irr").mean().alias("mean_irr"),
        pl.col("irr").sum().alias("total_irr"),
        pl.len().alias("n_users"),
    ])
    .with_columns([
        (pl.col("hit_rate_pct") * 100).round(4).alias("hit_rate_pct"),
        pl.col("mean_irr").round(6),
    ])
    .sort(["segment_label", "experimental_arm"])
)

# Export
ab_arm_summary.write_parquet(DATA_DIR / "ab_arm_summary.parquet", compression="zstd")
seg_breakdown.write_parquet(DATA_DIR / "ab_segment_breakdown.parquet", compression="zstd")

print(f"  ab_arm_summary.parquet       : {ab_arm_summary.height} rows (arm-level KPIs)")
print(f"  ab_segment_breakdown.parquet : {seg_breakdown.height} rows (segment × arm)")
print("PowerBI A/B export complete.")

Exporting A/B Test summary artifacts for PowerBI...
  Reusing user_metrics_pd from Section 4 execution.
  ab_arm_summary.parquet       : 3 rows (arm-level KPIs)
  ab_segment_breakdown.parquet : 15 rows (segment × arm)
PowerBI A/B export complete.


In [9]:
# =====================================================================
# POWERBI EXPORT: Virtual Assistant lookup table (Tab 4)
# =====================================================================
import json
import polars as pl
from pathlib import Path

# --- 1. Path Resolution ---
cwd = Path.cwd()
if (cwd / "data" / "processed").exists():
    DATA_DIR = cwd / "data" / "processed"
else:
    DATA_DIR = cwd.parent / "data" / "processed"

print("Building Virtual Assistant Data for PowerBI...")

# --- 2. Reverse Map IDs safely to Strings ---
with open(DATA_DIR / "id_mapping.json") as f:
    mapping = json.load(f)

# Map internal integer ID back to the original string ID
reverse_art = {int(v): str(k) for k, v in mapping["articles"].items()}

# --- 3. Build Article Metadata with 10-Digit Zero Padding ---
art_df = pl.read_parquet(DATA_DIR / "articles_processed.parquet").with_columns(
    pl.col("article_id")
    .map_elements(lambda x: reverse_art.get(x, ""), return_dtype=pl.Utf8)
    .str.zfill(10) # SAFELY PAD TO 10 DIGITS
    .alias("original_article_id")
).select([
    "article_id", "original_article_id", "prod_name", "product_group_name",
    "index_name", "colour_group_name", "garment_group_name"
])

# --- 4. Load LightGBM Predictions (The Primary L2R Engine) ---
lgbm_recs = pl.read_parquet(DATA_DIR / "lgbm_predictions.parquet")

# --- 5. Join Predictions, Article Metadata, and User Segments ---
va_data = (
    lgbm_recs
    .join(art_df, on="article_id", how="left")
    .join(
        pl.read_parquet(DATA_DIR / "customers_segmented.parquet")
        .select(["customer_id", "segment_label", "age"]), 
        on="customer_id", 
        how="left"
    )
)

# --- 6. Create the Image Filename Column ---
va_data = va_data.with_columns(
    pl.format("{}.jpg", pl.col("original_article_id")).alias("image_filename")
)

# --- 7. Export ---
out_path = DATA_DIR / "virtual_assistant_data.parquet"
va_data.write_parquet(out_path, compression="zstd")

print(f"SUCCESS: {va_data.height:,} rows exported to {out_path.name}")
print("\n[PowerBI Instructions for Tab 4]")
print("1. Set 'customer_id' as a dropdown Slicer.")
print("2. Filter a gallery/table visual using 'rank' (1 to 12).")
print("3. Create a calculated column in PowerBI: ImagePath = \"file:///C:/[Your_Absolute_Path]/data/article_images/\" & [image_filename]")
print("4. Set ImagePath's Data Category to 'Image URL' to render the product photos.")

Building Virtual Assistant Data for PowerBI...
SUCCESS: 2,597,748 rows exported to virtual_assistant_data.parquet

[PowerBI Instructions for Tab 4]
1. Set 'customer_id' as a dropdown Slicer.
2. Filter a gallery/table visual using 'rank' (1 to 12).
3. Create a calculated column in PowerBI: ImagePath = "file:///C:/[Your_Absolute_Path]/data/article_images/" & [image_filename]
4. Set ImagePath's Data Category to 'Image URL' to render the product photos.
